Le best modele

## 1. Imports

In [54]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from hydrosense.interface.main import load_data
# Ajouter en haut de la cellule
from colorama import Fore, Style


In [56]:
from hydrosense.ml_logic.folding import get_folds

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from hydrosense.interface.main import split_data

In [ ]:
from hydrosense.interface.main import load_data, evaluate, evaluate_deeper, preprocess, train
from hydrosense.ml_logic.folding import get_folds
from hydrosense.preprocess.cleaning import clean_piezo
from hydrosense.database.bigquery import load_piezo_bq
from hydrosense import params


In [58]:
from hydrosense.database.bigquery import load_plean
DATA_CODE_PIEZO = "BSS001QHYH"
df = load_plean(DATA_CODE_PIEZO)

✅ BSS001QHYH : 11821 lignes de features ML chargées.


In [81]:
df

,bss_id,date_mesure,niveau_nappe_eau,RR_synth,TM_synth,FFM_synth,PU_synth,PC1,PC2,PC3
0,BSS002EDYK,1992-10-23,28.14,5.93,12.00,2.9,3.860,3.915226,0.464661,0.886064
1,BSS002EDYK,1992-10-24,28.18,12.11,12.50,3.4,9.859,3.948009,0.468551,0.893483
2,BSS002EDYK,1992-10-25,28.30,9.84,14.70,5.0,6.933,3.977514,0.472052,0.900160
3,BSS002EDYK,1992-10-26,28.38,5.14,13.20,1.9,3.397,4.003740,0.475164,0.906096
4,BSS002EDYK,1992-10-27,28.38,3.76,14.10,4.1,1.191,4.026688,0.477887,0.911289
...,...,...,...,...,...,...,...,...,...,...
12256,BSS002EDYK,2026-05-14,26.96,16.50,11.75,5.0,12.807,1.696960,0.659848,-0.138771
12257,BSS002EDYK,2026-05-15,26.97,4.03,11.11,3.2,0.667,1.673158,0.658903,-0.076227
12258,BSS002EDYK,2026-05-16,27.03,1.86,11.95,2.7,-1.534,1.759426,0.635670,-0.040679
12259,BSS002EDYK,2026-05-17,27.05,1.96,13.85,2.6,-1.661,1.885524,0.591335,-0.037464


In [91]:
from hydrosense.database.bigquery import load_pem_bq, info_piezo
from hydrosense.preprocess.preprocessor import preprocess_week,preprocess_week_w_PU_synth
from hydrosense.utils.evap import estim_PU


Loading XGBoost...

✅ XGBoost loaded (0.0s)


In [92]:
from hydrosense.interface.main import preprocess_slim
df_week = preprocess_week_w_PU_synth(df)
X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week)



⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 1701 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 1636 mois | Test : 13 mois



In [93]:
df_week

,niveau_nappe_eau,semaine_sin,semaine_cos,lag_1,lag_2,lag_3,lag_4,lag_52,PC1_lag_1,PC2_lag_1,PC3_lag_1,PU_synth_lag_1,PU_synth_lag_2,PU_synth_lag_3,PU_synth_lag_4
date_mesure,,,,,,,,,,,,,,,
1993-10-24,26.784286,-0.935016,0.354605,27.024286,27.042857,26.524286,26.132857,28.206667,2.534364,0.300811,0.573558,0.759429,-0.566714,8.116000,10.995571
1993-10-31,26.735714,-0.885456,0.464723,26.784286,27.024286,27.042857,26.524286,28.477143,2.600164,0.308618,0.588449,0.128286,0.759429,-0.566714,8.116000
1993-11-07,26.788571,-0.822984,0.568065,26.735714,26.784286,27.024286,27.042857,28.151429,3.201027,0.379916,0.724432,-0.938857,0.128286,0.759429,-0.566714
1993-11-14,26.825714,-0.748511,0.663123,26.788571,26.735714,26.784286,27.024286,27.601429,3.583417,0.425289,0.810971,2.612143,-0.938857,0.128286,0.759429
1993-11-21,26.844286,-0.663123,0.748511,26.825714,26.788571,26.735714,26.784286,28.117143,3.919441,0.465161,0.887018,-0.385571,2.612143,-0.938857,0.128286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-26,27.045714,0.885456,-0.464723,27.118571,27.151429,27.201429,27.278571,27.065714,2.180096,1.912442,-0.826843,-2.545429,-1.234000,-1.185714,-2.054143
2026-05-03,27.011429,0.822984,-0.568065,27.045714,27.118571,27.151429,27.201429,27.104286,1.826535,1.884814,-0.736395,-3.108571,-2.545429,-1.234000,-1.185714
2026-05-10,27.047143,0.748511,-0.663123,27.011429,27.045714,27.118571,27.151429,27.108571,1.204911,1.720551,-0.637630,1.646857,-3.108571,-2.545429,-1.234000


In [94]:
# TRAIN / VAL Folding
splits_ml = get_folds(X_train_df.index, n_splits=25, min_train_years=3 , val_months_duration= 3)

In [95]:
def cal_rmsse(y_true, y_pred, y_train_hist):
    # 1. Erreur quadratique moyenne de TON modèle sur le test (Le haut)
    numerator = np.mean((y_true - y_pred) ** 2)

    # 2. Erreur quadratique moyenne de la persistance sur le train (Le bas)
    # np.diff(y_train_hist) fait la soustraction entre la semaine T et la semaine T-1
    denominator = np.mean(np.diff(y_train_hist) ** 2)

    # 3. On divise, et on applique la racine carrée (Root)
    return np.sqrt(numerator / denominator)

In [96]:
df_week

,niveau_nappe_eau,semaine_sin,semaine_cos,lag_1,lag_2,lag_3,lag_4,lag_52,PC1_lag_1,PC2_lag_1,PC3_lag_1,PU_synth_lag_1,PU_synth_lag_2,PU_synth_lag_3,PU_synth_lag_4
date_mesure,,,,,,,,,,,,,,,
1993-10-24,26.784286,-0.935016,0.354605,27.024286,27.042857,26.524286,26.132857,28.206667,2.534364,0.300811,0.573558,0.759429,-0.566714,8.116000,10.995571
1993-10-31,26.735714,-0.885456,0.464723,26.784286,27.024286,27.042857,26.524286,28.477143,2.600164,0.308618,0.588449,0.128286,0.759429,-0.566714,8.116000
1993-11-07,26.788571,-0.822984,0.568065,26.735714,26.784286,27.024286,27.042857,28.151429,3.201027,0.379916,0.724432,-0.938857,0.128286,0.759429,-0.566714
1993-11-14,26.825714,-0.748511,0.663123,26.788571,26.735714,26.784286,27.024286,27.601429,3.583417,0.425289,0.810971,2.612143,-0.938857,0.128286,0.759429
1993-11-21,26.844286,-0.663123,0.748511,26.825714,26.788571,26.735714,26.784286,28.117143,3.919441,0.465161,0.887018,-0.385571,2.612143,-0.938857,0.128286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-26,27.045714,0.885456,-0.464723,27.118571,27.151429,27.201429,27.278571,27.065714,2.180096,1.912442,-0.826843,-2.545429,-1.234000,-1.185714,-2.054143
2026-05-03,27.011429,0.822984,-0.568065,27.045714,27.118571,27.151429,27.201429,27.104286,1.826535,1.884814,-0.736395,-3.108571,-2.545429,-1.234000,-1.185714
2026-05-10,27.047143,0.748511,-0.663123,27.011429,27.045714,27.118571,27.151429,27.108571,1.204911,1.720551,-0.637630,1.646857,-3.108571,-2.545429,-1.234000


In [97]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor

all_results = []  # Stocke les résultats de chaque piézo

for bss in params.TARGETS_BSS:
    print(f"\n{'═'*55}")
    print(f"🔄 Traitement : {bss}")
    print(f"{'═'*55}")

    try:
        # ── Chargement & preprocessing ───────────────────────────
        df = load_plean(bss)
        df_week = preprocess_week_w_PU_synth(df)
        X_train_df, X_test_df, y_train_df, y_test_df = split_data(df_week)

        print(f"   Train : {len(X_train_df)} sem. | Test : {len(X_test_df)} sem.")

        # ── Baseline A : Persistance immédiate ───────────────────
        valeur_derniere = y_train_df.iloc[-1]
        y_pred_baseline_hebdo = np.full(len(y_test_df), fill_value=valeur_derniere)

        # ── Baseline B : Persistance saisonnière (J-52) ──────────
        y_pred_baseline_saisonniere = []
        for date in y_test_df.index:
            date_an_dernier = date - pd.Timedelta(weeks=52)
            idx = y_train_df.index.get_indexer([date_an_dernier], method='nearest')[0]
            y_pred_baseline_saisonniere.append(y_train_df.iloc[idx])
        y_pred_baseline_saisonniere = np.array(y_pred_baseline_saisonniere)

        # ── SARIMAX ───────────────────────────────────────────────
        try:
            model_sarimax = SARIMAX(
                y_train_df,
                order=(1, 0, 1),
                seasonal_order=(1, 0, 0, 52),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            preds_sarimax = model_sarimax.fit(disp=False).predict(
                start=len(y_train_df),
                end=len(y_train_df) + len(y_test_df) - 1
            ).values
        except Exception as e:
            print(f"   ⚠️ SARIMAX échoué ({e}) — remplacé par NaN")
            preds_sarimax = np.full(len(y_test_df), np.nan)

        # ── XGBoost best params ───────────────────────────────────
        best_xgb = XGBRegressor(
            colsample_bytree=0.6, learning_rate=0.1, max_depth=2,
            min_child_weight=5, n_estimators=500, subsample=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )
        best_xgb.fit(X_train_df, y_train_df)

        # ── Dictionnaire des modèles ──────────────────────────────
        models = {
            'Baseline (Semaine Dernière)': y_pred_baseline_hebdo,
            'Baseline (Année Dernière)':   y_pred_baseline_saisonniere,
            'SARIMAX(1,0,1)(1,0,0)[52]':   preds_sarimax,
            'Lasso':                        Lasso(alpha=0.01, random_state=42),
            'ElasticNet':                   ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42),
            'Random Forest':                RandomForestRegressor(n_estimators=300, max_depth=5,random_state=42, n_jobs=-1),
            'XGBoost Optimisé':             best_xgb,
        }

        # ── Calcul des métriques ──────────────────────────────────
        for name, m in models.items():
            if isinstance(m, np.ndarray):
                preds = m
            else:
                if name != 'XGBoost Optimisé':
                    m.fit(X_train_df, y_train_df)
                preds = m.predict(X_test_df)

            if np.any(np.isnan(preds)):
                mae, rmsse = np.nan, np.nan
            else:
                mae   = mean_absolute_error(y_test_df, preds)
                rmsse = cal_rmsse(y_test_df, preds, y_train_df)

            all_results.append({
                'BSS':         bss,
                'Modèle':      name,
                'MAE (m NGF)': round(mae, 4) if not np.isnan(mae) else np.nan,
                'RMSSE':       round(rmsse, 4) if not np.isnan(rmsse) else np.nan,
                'n_test':      len(y_test_df),
            })

        print(f"   ✅ OK")

    except Exception as e:
        print(f"   ❌ Erreur sur {bss} : {e}")
        continue

# ── Tableau global ────────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_results)

# Vue synthétique : MAE moyenne par modèle sur tous les piézos
df_summary = (
    df_all.groupby('Modèle')[['MAE (m NGF)', 'RMSSE']]
    .mean()
    .round(4)
    .sort_values('MAE (m NGF)')
)

print("\n\n🏆 SYNTHÈSE TOUS PIÉZOS — MAE & RMSSE moyens")
print("═" * 55)
display(df_summary.style.background_gradient(cmap='RdYlGn_r'))

# Vue détaillée : tous les piézos × tous les modèles
df_pivot = df_all.pivot(index='BSS', columns='Modèle', values='MAE (m NGF)').round(4)
print("\n📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE")
display(df_pivot.style.background_gradient(cmap='RdYlGn_r', axis=1))


═══════════════════════════════════════════════════════
🔄 Traitement : BSS002DEZW
═══════════════════════════════════════════════════════
✅ BSS002DEZW : 10370 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 0 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 0 mois | Test : 0 mois

   Train : 0 sem. | Test : 0 sem.
   ❌ Erreur sur BSS002DEZW : single positional indexer is out-of-bounds

═══════════════════════════════════════════════════════
🔄 Traitement : BSS000ZPHJ
═══════════════════════════════════════════════════════
✅ BSS000ZPHJ : 7464 lignes de features ML chargées.

⭐️ Use case: preprocess_week_w_PU_synth
✅ preprocess() done — 933 semaines | 15 colonnes


⭐️ Use case: split_data
✅ split_data() done — Train : 867 mois | Test : 13 mois

   Train : 867 sem. | Test : 13 sem.
   ✅ OK

═══════════════════════════════════════════════════════
🔄 Traitement : BSS000ZQXN
══════════════════════════════════════════

,MAE (m NGF),RMSSE
Modèle,,
ElasticNet,0.137000,0.643000
Lasso,0.137700,0.659000
Random Forest,0.144600,0.702200
XGBoost Optimisé,0.145800,0.678100
"SARIMAX(1,0,1)(1,0,0)[52]",0.820200,2.569500
Baseline (Semaine Dernière),0.856400,2.758600
Baseline (Année Dernière),1.510000,4.695000



📊 MAE DÉTAILLÉE PAR PIÉZO ET MODÈLE


Modèle,Baseline (Année Dernière),Baseline (Semaine Dernière),ElasticNet,Lasso,Random Forest,"SARIMAX(1,0,1)(1,0,0)[52]",XGBoost Optimisé
BSS,,,,,,,
BSS000ZPHJ,0.471200,0.756900,0.119400,0.110000,0.086400,0.657200,0.087800
BSS000ZQXN,0.863700,1.358900,0.134400,0.134300,0.156400,1.231000,0.122300
BSS001PGUQ,1.009000,0.707600,0.070600,0.070200,0.085800,0.633600,0.067600
BSS001QHPU,1.100800,0.939300,0.116900,0.114800,0.105000,0.598200,0.080500
BSS001QHYH,0.509900,0.391300,0.088900,0.093000,0.062200,0.248900,0.081700
BSS001QSMT,1.531600,0.710200,0.089700,0.087700,0.118300,0.804800,0.075000
BSS001QTKG,5.216300,3.803400,0.231800,0.235900,0.321200,3.821800,0.139800
BSS001RQQE,1.115200,0.684000,0.098800,0.099900,0.148300,0.582500,0.158600
BSS001SHNE,2.062600,1.506300,0.211800,0.208400,0.176400,1.291200,0.235400


In [98]:
df_week

,niveau_nappe_eau,semaine_sin,semaine_cos,lag_1,lag_2,lag_3,lag_4,lag_52,PC1_lag_1,PC2_lag_1,PC3_lag_1,PU_synth_lag_1,PU_synth_lag_2,PU_synth_lag_3,PU_synth_lag_4
date_mesure,,,,,,,,,,,,,,,
1993-10-24,26.784286,-0.935016,0.354605,27.024286,27.042857,26.524286,26.132857,28.206667,2.534364,0.300811,0.573558,0.759429,-0.566714,8.116000,10.995571
1993-10-31,26.735714,-0.885456,0.464723,26.784286,27.024286,27.042857,26.524286,28.477143,2.600164,0.308618,0.588449,0.128286,0.759429,-0.566714,8.116000
1993-11-07,26.788571,-0.822984,0.568065,26.735714,26.784286,27.024286,27.042857,28.151429,3.201027,0.379916,0.724432,-0.938857,0.128286,0.759429,-0.566714
1993-11-14,26.825714,-0.748511,0.663123,26.788571,26.735714,26.784286,27.024286,27.601429,3.583417,0.425289,0.810971,2.612143,-0.938857,0.128286,0.759429
1993-11-21,26.844286,-0.663123,0.748511,26.825714,26.788571,26.735714,26.784286,28.117143,3.919441,0.465161,0.887018,-0.385571,2.612143,-0.938857,0.128286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-26,27.045714,0.885456,-0.464723,27.118571,27.151429,27.201429,27.278571,27.065714,2.180096,1.912442,-0.826843,-2.545429,-1.234000,-1.185714,-2.054143
2026-05-03,27.011429,0.822984,-0.568065,27.045714,27.118571,27.151429,27.201429,27.104286,1.826535,1.884814,-0.736395,-3.108571,-2.545429,-1.234000,-1.185714
2026-05-10,27.047143,0.748511,-0.663123,27.011429,27.045714,27.118571,27.151429,27.108571,1.204911,1.720551,-0.637630,1.646857,-3.108571,-2.545429,-1.234000
